# MS3 — Step 7: Structure-Affinity Self-Consistency

Tests whether PRODIGY's independently-derived binding affinity (Step 6) correlates with
Step 5's independently-derived structural groove-geometry features, computed from the
same 35 AF3-predicted structures. This is an internal-consistency check, not a validation
against external immunogenicity data.

**Why this design, not an external immunogenicity correlation**: the original design
intended to correlate PRODIGY affinity against a published SARS-CoV-2 immunogenicity
dataset (Zornikova/Efimov et al., Communications Biology 2022). Direct verification of
that paper found it tested only 4 alleles (HLA-A*02:01, A*03:01, B*40:01, B*27:05), while
this manuscript's panel spans 16 alleles including all 6 DRB1 class II alleles and B*27:05
is not even in the panel. Using that paper as ground truth would have provided real
correlation data for only a handful of the 35 pairs and none for the class II arm,
while silently implying full-panel coverage. This step instead tests something the
pipeline's own two independent methods (PyMOL-derived geometry, PRODIGY-derived
energetics) can genuinely speak to across all 35 pairs: physical plausibility and
self-consistency between structure and predicted energetics.

**Class I and class II are tested separately, not pooled.** Class II's joint DRA+DRB1
contact surface is structurally different in kind (larger, two-chain) from class I's
single-chain contact (Step 5 confirmed this directly: DRB1 pairs show higher contact
counts and BSA throughout). Pooling both into one correlation risks a result driven by
the class I/II split itself rather than a genuine within-type structure-affinity
relationship.

**Correlations tested**, each run separately for class I and class II:
- Buried surface area (BSA) vs. PRODIGY ΔG
- Contact residue count vs. PRODIGY ΔG
- Polar contact count vs. PRODIGY ΔG

**Statistics:** Spearman rank correlation (rank-based, does not assume linearity,
appropriate given the small per-group sample sizes: ~23 class I, ~12 class II).

## Part A — Load Step 5 and Step 6 Tables

In [ ]:
!pip install -q pandas scipy 2>/dev/null || pip install -q --break-system-packages pandas scipy

from pathlib import Path
from typing import Optional
import shutil

import numpy as np
import pandas as pd
from scipy import stats

BASE_DIR = Path("/content/ms3_step7_workspace")
TABLES_DIR = BASE_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

STEP5_KNOWN_PATH = Path("/content/ms3_step5_workspace/tables/Table_S2_groove_geometry_concordance.csv")
STEP6_KNOWN_PATH = Path("/content/ms3_step6_workspace/tables/Table_S3_prodigy_affinity.csv")

def load_or_upload(known_path: Path, label: str, name_hint: str) -> pd.DataFrame:
    """
    Loads a table from its known prior-step location if this session already
    has it (e.g. Steps 5/6 ran earlier in the same runtime), otherwise
    prompts for upload. /content/ is ephemeral and does not persist across
    separate Colab sessions, so a fresh runtime needs these re-uploaded even
    if they were already generated once.
    """
    local_copy = BASE_DIR / f"{name_hint}.csv"
    if known_path.exists() and not local_copy.exists():
        shutil.copy(known_path, local_copy)
        print(f"Found {label} already in this session -- copied from {known_path}")
    elif local_copy.exists():
        print(f"{label} already present at {local_copy}")
    else:
        try:
            from google.colab import files
            print(f"{label} not found automatically -- upload it now:")
            uploaded = files.upload()
            for fname, content in uploaded.items():
                (BASE_DIR / fname).write_bytes(content)
            candidates = list(BASE_DIR.glob(f"*{name_hint.split(chr(95))[0]}*.csv")) or list(BASE_DIR.glob("*.csv"))
            if candidates and not local_copy.exists():
                shutil.copy(candidates[0], local_copy)
        except ImportError:
            pass

    if not local_copy.exists():
        raise SystemExit(f"[BLOCKED] {label} not available -- cannot proceed without it.")
    return pd.read_csv(local_copy)

step5_df = load_or_upload(STEP5_KNOWN_PATH, "Step 5 groove geometry table", "step5_geometry")
step6_df = load_or_upload(STEP6_KNOWN_PATH, "Step 6 PRODIGY affinity table", "step6_prodigy")

print(f"\nStep 5 table: {len(step5_df)} rows, columns: {list(step5_df.columns)}")
print(f"Step 6 table: {len(step6_df)} rows, columns: {list(step6_df.columns)}")

In [ ]:
# Merge on pair_id -- Step 6's table may already include Step 5's columns if
# it was run after Step 5 in the same session, so avoid duplicate columns
step6_affinity_only = step6_df[["pair_id", "delta_g_kcal_mol", "kd_M"]].copy()

merged_df = step5_df.merge(step6_affinity_only, on="pair_id", how="inner", suffixes=("", "_dup"))
merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith("_dup")]

print(f"Merged: {len(merged_df)} pairs with both geometry and affinity data")

required_cols = ["buried_surface_area_A2", "n_contact_residues", "n_polar_contacts",
                  "delta_g_kcal_mol", "is_class_ii"]
missing_cols = [c for c in required_cols if c not in merged_df.columns]
if missing_cols:
    raise SystemExit(f"[BLOCKED] Merged table is missing required column(s): {missing_cols}. "
                     f"Check that Step 5 and Step 6 both completed successfully.")

usable_df = merged_df.dropna(subset=required_cols).copy()
print(f"Usable rows (no missing values in required columns): {len(usable_df)}")
print(f"\nClass breakdown:")
print(usable_df["is_class_ii"].value_counts().rename({True: "Class II", False: "Class I"}).to_string())

usable_df.head(10)

## Part B — Structure-Affinity Correlations, Class I and Class II Separately

In [ ]:
def run_correlations(df: pd.DataFrame, group_label: str) -> list:
    """Spearman correlation of each structural feature against PRODIGY dG,
    within one class (I or II) at a time."""
    feature_cols = {
        "buried_surface_area_A2": "Buried surface area (A2)",
        "n_contact_residues": "Contact residue count",
        "n_polar_contacts": "Polar contact count",
    }
    results = []
    n = len(df)
    for col, label in feature_cols.items():
        if n < 4:
            results.append({"group": group_label, "feature": label, "n": n,
                            "spearman_r": None, "p_value": None,
                            "note": "n<4, correlation not meaningfully interpretable"})
            continue
        rho, p = stats.spearmanr(df[col], df["delta_g_kcal_mol"])
        results.append({"group": group_label, "feature": label, "n": n,
                        "spearman_r": round(rho, 3), "p_value": round(p, 4),
                        "note": ""})
    return results

class_i_df = usable_df[usable_df["is_class_ii"] == False]
class_ii_df = usable_df[usable_df["is_class_ii"] == True]

print(f"Class I: n={len(class_i_df)}")
print(f"Class II: n={len(class_ii_df)}")

all_results = run_correlations(class_i_df, "Class I") + run_correlations(class_ii_df, "Class II")
results_df = pd.DataFrame(all_results)

print("\nCorrelation results (Spearman rho, negative = more favorable dG with larger/more feature, as expected):")
print(results_df.to_string(index=False))

# Flag any correlation in the physically UNEXPECTED direction -- BSA/contacts/polar
# contacts should generally correlate NEGATIVELY with dG (more burial -> more
# favorable, i.e. more negative, binding energy). A positive rho here would be
# a real finding worth investigating, not just a low-N noise artifact to wave away.
for _, row in results_df.iterrows():
    r_val = row["spearman_r"]
    p_val = row["p_value"]
    if r_val is not None and r_val > 0 and p_val is not None and p_val < 0.1:
        group = row["group"]
        feature = row["feature"]
        print(f"\n[FLAG] {group} / {feature}: positive correlation "
              f"(rho={r_val}, p={p_val}) -- this is the physically "
              f"unexpected direction (more burial/contacts predicting LESS favorable binding) "
              f"and is worth inspecting the underlying pairs directly rather than dismissing.")

results_df

## Part C — Scatter Data Export (for manuscript figure generation)

Exports the raw paired values behind each correlation, so the actual scatter plots can
be generated in your preferred plotting environment with full control over styling.

In [ ]:
scatter_export = usable_df[[
    "pair_id", "allele", "locus", "peptide", "is_class_ii",
    "buried_surface_area_A2", "n_contact_residues", "n_polar_contacts",
    "delta_g_kcal_mol", "kd_M",
]].copy()

scatter_path = TABLES_DIR / "Table_S4_structure_affinity_scatter_data.csv"
scatter_export.to_csv(scatter_path, index=False)

results_path = TABLES_DIR / "Table_S4_structure_affinity_correlations.csv"
results_df.to_csv(results_path, index=False)

print(f"Saved scatter data: {scatter_path}")
print(f"Saved correlation results: {results_path}")

try:
    from google.colab import files
    files.download(str(scatter_path))
    files.download(str(results_path))
except ImportError:
    print(f"Tables saved locally.")

print("\nStep 7 complete.")
print("Interpretation guide: a significant negative Spearman rho for BSA/contacts vs "
      "dG within a class supports physical plausibility of the AF3+PRODIGY pipeline "
      "(better-buried, more-contacted interfaces predict more favorable binding, as "
      "expected from first principles). Absence of significance at this sample size "
      "(n~12-23 per class) is not strong evidence against the pipeline -- state this "
      "honestly as a power limitation in the manuscript rather than over-interpreting "
      "a null result.")